In [1]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchvision.transforms.functional as tvF

from mpl_toolkits.axes_grid1 import make_axes_locatable

import math

In [2]:
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, num_hiddens, dropout, max_len=1000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(dropout)
        self.P = torch.zeros((1, max_len, num_hiddens))
        X = torch.arange(max_len, dtype=torch.float32).reshape(-1, 1) / torch.pow(10000, torch.arange(0, num_hiddens, 2, dtype=torch.float32) / num_hiddens)
        self.P[:, :, 0::2] = torch.sin(X)
        self.P[:, :, 1::2] = torch.cos(X[:,0:2])

    def forward(self, X):
        # plt.figure(dpi=200)
        # plt.imshow(self.P[0,:,:].detach().cpu().numpy())
        # plt.show()
        # print(self.P.size())
        # print(X.size())
        X = X + self.P[:, :X.shape[1], :].to(X.device)
        return self.dropout(X)

class ConvEmbeding(nn.Module):
    def __init__(self, TTE_args):
        super(ConvEmbeding, self).__init__()
        self.Feature_detector = nn.Sequential(
            nn.Conv1d(in_channels=TTE_args['input_channels'], out_channels=TTE_args['expected_dim'],
                    kernel_size=TTE_args['kernel_size'], stride=TTE_args['stride']),
            nn.ReLU(),
        )
        self.Norm_Pool = nn.Sequential(
            nn.BatchNorm1d(num_features=TTE_args['expected_dim'], affine=False),
            nn.MaxPool1d(kernel_size = TTE_args['maxpool_size']),
        )
        self.Embeding = nn.Sequential(
            self.Feature_detector,
            self.Norm_Pool,
        )

    def forward(self, x):
        return self.Embeding(x).permute(0,2,1)

class TemporalTransformerEncoder(nn.Module):
    def __init__(self, TTE_args):
        super(TemporalTransformerEncoder, self).__init__()
        self.Embeding = ConvEmbeding(TTE_args)
        self.PositionalEncoding = PositionalEncoding(TTE_args['expected_dim'], TTE_args['position_dropout'])
        self.transformer_encoder = nn.TransformerEncoder(nn.TransformerEncoderLayer(TTE_args['expected_dim'], TTE_args['num_heads'], batch_first = True), TTE_args['num_layers'])
        self.FC = nn.Sequential(
            nn.Flatten(start_dim = 1),
            nn.LazyLinear(out_features = TTE_args['category_num']),
        )

    def forward(self, x):
        # print(self.Embeding(x).size())
        # print(self.PositionalEncoding(self.Embeding(x)).size())
        # print(self.transformer_encoder(self.PositionalEncoding(self.Embeding(x))).size())
        return F.log_softmax(self.FC(self.transformer_encoder(self.PositionalEncoding(x))),dim=1)
        # return F.log_softmax(self.FC(self.transformer_encoder(self.Embeding(x))),dim=1)

def kernel_init(run_TTE, TTE_args):
    for param in run_TTE.Embeding.Feature_detector.parameters():
        param.requires_grad = False
    run_TTE_param = run_TTE.state_dict()
    step = TTE_args['kernel_size'] // 6 // TTE_args['expected_dim']
    for ii in range(0, TTE_args['expected_dim']):
        run_TTE_param['Embeding.Feature_detector.0.weight'][ii] = torch.tensor(signal.windows.gaussian(TTE_args['kernel_size'],2 + ii * step), dtype=dtype).to(device)
    run_TTE_param['Embeding.Feature_detector.0.bias'][:] = 0.0
    run_TTE.load_state_dict(run_TTE_param)
    return run_TTE

def acu_cal(log_py_train, y_train, log_py_test, y_test, acu):
    acu['acu_train'].append((torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0)).item())
    acu['acu_test'].append((torch.sum(torch.argmax(log_py_test, dim=1) == y_test) / y_test.size(0)).item())
    acu['acu_train_STDP'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 0)] == y_train[torch.argwhere(y_train == 0)]) / y_train[torch.argwhere(y_train == 0)].size(0)).item())
    acu['acu_test_STDP'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 0)] == y_test[torch.argwhere(y_test == 0)]) / y_test[torch.argwhere(y_test == 0)].size(0)).item())
    acu['acu_train_BurstProp'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 1)] == y_train[torch.argwhere(y_train == 1)]) / y_train[torch.argwhere(y_train == 1)].size(0)).item())
    acu['acu_test_BurstProp'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 1)] == y_test[torch.argwhere(y_test == 1)]) / y_test[torch.argwhere(y_test == 1)].size(0)).item())
    acu['acu_train_BP1'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 2)] == y_train[torch.argwhere(y_train == 2)]) / y_train[torch.argwhere(y_train == 2)].size(0)).item())
    acu['acu_test_BP1'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 2)] == y_test[torch.argwhere(y_test == 2)]) / y_test[torch.argwhere(y_test == 2)].size(0)).item())
    acu['acu_train_BP2'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 3)] == y_train[torch.argwhere(y_train == 3)]) / y_train[torch.argwhere(y_train == 3)].size(0)).item())
    acu['acu_test_BP2'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 3)] == y_test[torch.argwhere(y_test == 3)]) / y_test[torch.argwhere(y_test == 3)].size(0)).item())
    acu['acu_train_NC'].append((torch.sum(torch.argmax(log_py_train, dim=1)[torch.argwhere(y_train == 4)] == y_train[torch.argwhere(y_train == 4)]) / y_train[torch.argwhere(y_train == 4)].size(0)).item())
    acu['acu_test_NC'].append((torch.sum(torch.argmax(log_py_test, dim=1)[torch.argwhere(y_test == 4)] == y_test[torch.argwhere(y_test == 4)]) / y_test[torch.argwhere(y_test == 4)].size(0)).item())
    return acu

In [4]:
def confusion_cal(y_test, py_test, y_type):
    class_num = len(y_test.unique())
    confusion_num = np.zeros((class_num, class_num))
    # y_test = torch.tensor(y_test)
    # py_test = torch.tensor(py_test)
    if y_type == 0:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(torch.argmax(py_test, dim=1) == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))
    elif y_type == 1:
        for ii in range(0, class_num):
            for jj in range(0, class_num):
                confusion_num[ii,jj] = len(np.intersect1d(torch.argwhere(py_test == jj).detach().cpu().numpy(), torch.argwhere(y_test == ii).detach().cpu().numpy()))

    return confusion_num

In [5]:
acu_set = np.zeros((50,12))
acu_set_test = np.zeros((50,12))
dat_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/data'
SPK = []
ptype = []
for rnum in range(0, 30):
    SPK.append(np.load(dat_dir + f'/FixedRepresentation/Spk_an_wn_{rnum}.npy'))
    ptype.append(np.load(dat_dir + f'/FixedRepresentation/Ptype_an_wn_{rnum}.npy'))

SPK = np.concatenate(SPK, axis=0)
ptype = np.concatenate(ptype, axis=0)

SPK1 = []
ptype1 = []
for rnum in range(0, 49):
    SPK1.append(np.load(dat_dir + f'/FixedRepresentation01/Spk_an_wn_{rnum}.npy'))
    ptype1.append(np.load(dat_dir + f'/FixedRepresentation01/Ptype_an_wn_{rnum}.npy'))

SPK1 = np.concatenate(SPK1, axis=0)
ptype1 = np.concatenate(ptype1, axis=0)

for ii in range(0,50):
    train_ind = np.random.choice(np.size(SPK,axis=0), np.int32(0.8 * np.size(SPK,axis=0)), replace=False)
    test_ind = np.setdiff1d(np.arange(0, np.size(SPK,axis=0)), train_ind)
    test_ind_od = np.random.choice(np.size(SPK1,axis=0), np.int32(0.2 * np.size(SPK1,axis=0)), replace=False)

    x_train = torch.tensor(SPK[train_ind,:], dtype=dtype).to(device).unsqueeze(dim=1)
    y_train = torch.tensor(ptype[train_ind], dtype=torch.long).to(device)

    x_test = torch.tensor(SPK[test_ind,:], dtype=dtype).to(device).unsqueeze(dim=1)
    y_test = torch.tensor(ptype[test_ind], dtype=torch.long).to(device)

    x_test_od = torch.tensor(SPK1[test_ind_od,:], dtype=dtype).to(device).unsqueeze(dim=1)
    y_test_od = torch.tensor(ptype1[test_ind_od], dtype=torch.long).to(device)

    TTE_args = {
        'input_channels':1,
        'kernel_size':500,
        'stride':1,
        'maxpool_size':125,
        'num_heads':5,
        'expected_dim':5,
        'num_layers':1,
        'category_num':5,
        'position_dropout':0.0,
        'data_len':x_train.size(dim=1),
    }

    acu = {
        'acu_train':[],
        'acu_test':[],
        'acu_train_STDP':[],
        'acu_test_STDP':[],
        'acu_train_BurstProp':[],
        'acu_test_BurstProp':[],
        'acu_train_BP1':[],
        'acu_test_BP1':[],
        'acu_train_BP2':[],
        'acu_test_BP2':[],
        'acu_train_NC':[],
        'acu_test_NC':[],
    }
    acu_test = {
        'acu_train':[],
        'acu_test':[],
        'acu_train_STDP':[],
        'acu_test_STDP':[],
        'acu_train_BurstProp':[],
        'acu_test_BurstProp':[],
        'acu_train_BP1':[],
        'acu_test_BP1':[],
        'acu_train_BP2':[],
        'acu_test_BP2':[],
        'acu_train_NC':[],
        'acu_test_NC':[],
    }

    run_TTE = TemporalTransformerEncoder(TTE_args).to(device)
    run_TTE = kernel_init(run_TTE, TTE_args)
    loss_fn = nn.NLLLoss()
    optimizer = torch.optim.Adam(run_TTE.parameters(),lr=1e-3)

    run_TTE.train()

    x_train1 = run_TTE.Embeding(x_train)
    x_test1 = run_TTE.Embeding(x_test)

    x_test_od1 = run_TTE.Embeding(x_test_od)

    for epoch in range(0, 500):
        log_py_train = run_TTE(x_train1)
        L = loss_fn(log_py_train,y_train)
        optimizer.zero_grad()
        L.backward()
        optimizer.step()
        if torch.sum(torch.argmax(log_py_train, dim=1) == y_train) / y_train.size(0) > 0.85:
            break

    run_TTE.eval()
    log_py_test = run_TTE(x_test1)
    L_test = loss_fn(log_py_test,y_test)
    acu = acu_cal(log_py_train, y_train, log_py_test, y_test, acu)
    log_py_test_od = run_TTE(x_test_od1)
    acu_test = acu_cal(log_py_train, y_train, log_py_test_od, y_test_od, acu_test)

    acu_set[ii,0] = acu['acu_train'][0]
    acu_set[ii,1] = acu['acu_test'][0]
    acu_set[ii,2] = acu['acu_train_STDP'][0]
    acu_set[ii,3] = acu['acu_test_STDP'][0]
    acu_set[ii,4] = acu['acu_train_BurstProp'][0]
    acu_set[ii,5] = acu['acu_test_BurstProp'][0]
    acu_set[ii,6] = acu['acu_train_BP1'][0]
    acu_set[ii,7] = acu['acu_test_BP1'][0]
    acu_set[ii,8] = acu['acu_train_BP2'][0]
    acu_set[ii,9] = acu['acu_test_BP2'][0]
    acu_set[ii,10] = acu['acu_train_NC'][0]
    acu_set[ii,11] = acu['acu_test_NC'][0]

    acu_set_test[ii,0] = acu_test['acu_train'][0]
    acu_set_test[ii,1] = acu_test['acu_test'][0]
    acu_set_test[ii,2] = acu_test['acu_train_STDP'][0]
    acu_set_test[ii,3] = acu_test['acu_test_STDP'][0]
    acu_set_test[ii,4] = acu_test['acu_train_BurstProp'][0]
    acu_set_test[ii,5] = acu_test['acu_test_BurstProp'][0]
    acu_set_test[ii,6] = acu_test['acu_train_BP1'][0]
    acu_set_test[ii,7] = acu_test['acu_test_BP1'][0]
    acu_set_test[ii,8] = acu_test['acu_train_BP2'][0]
    acu_set_test[ii,9] = acu_test['acu_test_BP2'][0]
    acu_set_test[ii,10] = acu_test['acu_train_NC'][0]
    acu_set_test[ii,11] = acu_test['acu_test_NC'][0]

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.num_heads is odd
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


In [6]:
# np.save('./FixedRepresentation/data0/TTE_args_fixedrepresentation.npy',TTE_args)

save_dir = '/content/drive/MyDrive/Project/PlasticityDecoding/result'
np.save(save_dir + '/TTE_fixedrepresentation.npy',acu_set)
np.save(save_dir + '/TTE_fixedrepresentation1.npy',acu_set_test)
np.save(save_dir + '/TTE_fixedrepresentation_confusion_num.npy', confusion_cal(y_test, log_py_test, y_type=0))
np.save(save_dir + '/TTE_fixedrepresentation_confusion_num1.npy', confusion_cal(y_test_od, log_py_test_od, y_type=0))

In [7]:
np.round(acu_set.mean(axis=0), 3)

array([0.852, 0.779, 0.922, 0.896, 0.841, 0.7  , 0.708, 0.606, 0.812,
       0.73 , 0.976, 0.961])

In [8]:
np.round(acu_set_test.mean(axis=0), 3)

array([0.852, 0.702, 0.922, 0.788, 0.841, 0.677, 0.708, 0.385, 0.812,
       0.694, 0.976, 0.969])